# Time-Travel Debugging for Multi-Agent Workflows

**Multigen SDK — Notebook 20**

---

## What is time-travel debugging?

In a conventional debugger you can set a breakpoint, pause execution, and inspect the
state of your program at that exact moment. **Time-travel debugging** goes further: it
records the entire execution history as a series of immutable snapshots, then lets you
_rewind_ to any past point and re-run from there — with or without code changes.

The concept is well-known in distributed systems (event sourcing, Datomic's database
of facts) and is now indispensable for AI agent pipelines.

## Why does it matter for multi-agent workflows?

A multi-agent pipeline can span dozens of LLM calls, tool invocations, and data
transformations.  When something goes wrong — a model hallucinates, a data-cleaning
step silently drops a column, an analysis produces NaN — the failure is usually
**downstream** of its true cause.

Without time-travel debugging you must:
1. Re-run the entire pipeline from scratch (expensive).
2. Hope the non-deterministic model gives the same intermediate output.
3. Manually reconstruct what the state was at each step.

With time-travel debugging you can:
1. Inspect the exact input and output at every step.
2. Pinpoint the step where the data diverged from expectations.
3. Patch one agent and replay _only_ from that step — with the original upstream
   context frozen in place.
4. Export the full trace for observability dashboards.

## What this notebook covers

| Section | Topic |
|---------|-------|
| 1 | Setup and imports |
| 2 | Building a simple 3-step pipeline |
| 3 | Listing all captured steps |
| 4 | Inspecting state at a specific step |
| 5 | Diffing two steps |
| 6 | Replay from a step with a patched agent |
| 7 | Replay with patched input parameters |
| 8 | SQLite persistence across sessions |
| 9 | Exporting a full trace |
| 10 | Injecting a failure and replaying with a fix |
| 11 | Multi-workflow debugging |

All agents are pure in-memory mock functions.  No external APIs, MongoDB, or
Temporal are required.

## Section 1 — Setup and Imports

The Multigen SDK ships `WorkflowDebugger` alongside two snapshot-store backends:

- **`InMemorySnapshotStore`** — stores everything in a Python dict; ideal for tests
  and notebook exploration.  Data is lost when the kernel restarts.
- **`SQLiteSnapshotStore`** — writes snapshots to a local SQLite database file;
  survives kernel restarts and can be queried across sessions.

`WorkflowDebugger` accepts either store via its `store=` constructor argument.  If you
omit it, an `InMemorySnapshotStore` is created automatically.

We also import `FunctionAgent` — the simplest Multigen agent type.  It wraps any
Python callable (sync or async) as a named agent.

In [ ]:
import asyncio
import json
import os
import sys

# Make sure the SDK is on the path when running from the notebooks/ directory
sdk_path = os.path.join(os.path.dirname(os.path.abspath('.')), 'sdk')
if sdk_path not in sys.path:
    sys.path.insert(0, sdk_path)

# Also try relative path for common notebook layouts
for candidate in ['../sdk', 'sdk']:
    candidate = os.path.abspath(candidate)
    if os.path.isdir(candidate) and candidate not in sys.path:
        sys.path.insert(0, candidate)

from multigen import WorkflowDebugger, FunctionAgent, InMemorySnapshotStore, SQLiteSnapshotStore

print('WorkflowDebugger :', WorkflowDebugger)
print('FunctionAgent    :', FunctionAgent)
print('InMemoryStore    :', InMemorySnapshotStore)
print('SQLiteStore      :', SQLiteSnapshotStore)
print()
print('All imports successful.')

## Section 2 — Creating a Simple 3-Step Pipeline

We model a tiny analytics pipeline with three stages:

```
data_fetch  →  data_clean  →  data_analyze
```

Each stage is a `FunctionAgent` wrapping a mock lambda — no network calls.

### How `run_chain` works

`WorkflowDebugger.run_chain` accepts a list of `(agent, params)` tuples.
It executes the steps sequentially.  After every step it saves a `Snapshot` to the
store.  Each snapshot captures:

- `step_index` — position in the chain (0-based)
- `node_id` — agent name used as a human-readable identifier
- `agent` — the Python class name of the agent
- `input_params` — the dict the agent received
- `output` — the dict the agent returned
- `context` — all accumulated outputs from steps 0 … this step
- `status` — `"success"` or `"failed"`
- `duration_ms` — wall-clock time for this step
- `timestamp` — UTC ISO-8601 timestamp

Downstream steps receive a merged view of all prior outputs, so `data_clean` can
reference fields produced by `data_fetch`, and `data_analyze` can reference both.

In [ ]:
# ── Define three mock agents ─────────────────────────────────────────────────

# Step 0: simulate fetching raw rows from a database
data_fetch_agent = FunctionAgent(
    'data_fetch',
    fn=lambda ctx: {
        'raw_rows': [
            {'id': 1, 'value': 10.5, 'flag': None},
            {'id': 2, 'value': None, 'flag': 'ok'},
            {'id': 3, 'value': 7.2,  'flag': 'ok'},
            {'id': 4, 'value': 3.1,  'flag': 'ok'},
        ],
        'source': 'mock_db',
        'record_count': 4,
    }
)

# Step 1: remove rows with missing values, return clean rows
data_clean_agent = FunctionAgent(
    'data_clean',
    fn=lambda ctx: {
        'clean_rows': [
            r for r in ctx.get('raw_rows', [])
            if r.get('value') is not None and r.get('flag') == 'ok'
        ],
        'dropped_count': sum(
            1 for r in ctx.get('raw_rows', [])
            if r.get('value') is None or r.get('flag') != 'ok'
        ),
    }
)

# Step 2: compute basic statistics on the clean rows
data_analyze_agent = FunctionAgent(
    'data_analyze',
    fn=lambda ctx: {
        'mean': (
            sum(r['value'] for r in ctx.get('clean_rows', [])) /
            len(ctx.get('clean_rows', [1]))
        ),
        'count': len(ctx.get('clean_rows', [])),
        'max':   max((r['value'] for r in ctx.get('clean_rows', [])), default=None),
        'min':   min((r['value'] for r in ctx.get('clean_rows', [])), default=None),
        'threshold_pass': (
            sum(r['value'] for r in ctx.get('clean_rows', [])) /
            len(ctx.get('clean_rows', [1]))
        ) >= ctx.get('threshold', 5.0),
    }
)

# ── Create the debugger and run the chain ────────────────────────────────────

debugger = WorkflowDebugger()   # uses InMemorySnapshotStore by default

steps = [
    (data_fetch_agent,   {'threshold': 5.0}),   # params available throughout
    (data_clean_agent,   {}),
    (data_analyze_agent, {}),
]

results = await debugger.run_chain(steps, workflow_id='wf-001')

print('Pipeline finished.  Outputs per step:')
for i, out in enumerate(results):
    print(f'  step {i}: {out}')

## Section 3 — Listing All Captured Steps

`await debugger.list_steps(workflow_id)` returns every `Snapshot` recorded for that
workflow, sorted by `step_index`.

A `Snapshot` is a dataclass with these fields:

| Field | Type | Description |
|-------|------|-------------|
| `workflow_id` | str | Which workflow this snapshot belongs to |
| `step_index` | int | Position in the chain |
| `node_id` | str | Agent name (human-readable) |
| `agent` | str | Python class name of the agent |
| `status` | str | `"success"` or `"failed"` |
| `duration_ms` | float | Wall-clock time for this step |
| `timestamp` | str | UTC ISO-8601 |
| `input_params` | dict | What the agent received |
| `output` | dict | What the agent returned |
| `context` | dict | All accumulated outputs up to this step |
| `error` | str\|None | Exception message if status is `"failed"` |

The `context` field is particularly useful for time-travel: it contains the _full
world state_ at the moment this step completed.

In [ ]:
steps_summary = await debugger.list_steps('wf-001')

print(f'Total steps captured: {len(steps_summary)}')
print()

for snap in steps_summary:
    print(f'Step {snap.step_index}  |  node={snap.node_id!r:<14}  agent={snap.agent!r:<15}'
          f'  status={snap.status!r:<10}  duration={snap.duration_ms:.2f} ms')
    print(f'         input_params keys : {list(snap.input_params.keys())}')
    print(f'         output keys       : {list(snap.output.keys())}')
    print()

## Section 4 — Inspecting State at a Specific Step

`await debugger.get_snapshot(workflow_id, step=N)` retrieves the snapshot for step N.

The most powerful field is `context`: it contains every output accumulated up to and
including step N.  This means:

- At step 0 (`data_fetch`): context holds the raw rows.
- At step 1 (`data_clean`): context holds raw rows AND clean rows.
- At step 2 (`data_analyze`): context holds everything including statistics.

This is exactly what time-travel gives you — you can ask "what did the world look
like at the moment step 1 finished?" without re-running anything.

In [ ]:
# Inspect the snapshot at step 1 (data_clean)
snap = await debugger.get_snapshot('wf-001', step=1)

print('=== Snapshot at step 1 (data_clean) ===')
print(f'  workflow_id  : {snap.workflow_id}')
print(f'  step_index   : {snap.step_index}')
print(f'  node_id      : {snap.node_id}')
print(f'  agent        : {snap.agent}')
print(f'  status       : {snap.status}')
print(f'  duration_ms  : {snap.duration_ms}')
print(f'  timestamp    : {snap.timestamp}')
print()
print('--- input_params (what data_clean received) ---')
print(json.dumps(snap.input_params, indent=2))
print()
print('--- output (what data_clean returned) ---')
print(json.dumps(snap.output, indent=2))
print()
print('--- context (full accumulated state after step 1) ---')
print(json.dumps(snap.context, indent=2))

## Section 5 — Diffing Two Steps

`await debugger.diff(workflow_id, step_a=A, step_b=B)` returns a dict showing what
changed between the snapshots at step A and step B.

It compares: `input_params`, `output`, `context`, `status`, and `error`.  Only the
fields that **differ** appear in the result, each structured as:

```python
{
  "context": {
    "before": { ...state at step A... },
    "after":  { ...state at step B... },
  }
}
```

Diffing step 0 → step 2 shows everything that was added by the clean and analyze
steps: the `context` grew from raw rows to include clean rows, dropped count,
statistical outputs, and threshold pass/fail.

In [ ]:
diff = await debugger.diff('wf-001', step_a=0, step_b=2)

print('=== Diff: step 0 (data_fetch) → step 2 (data_analyze) ===')
print()

for field, change in diff.items():
    print(f'Field changed: {field!r}')
    if isinstance(change, dict) and 'before' in change:
        print(f'  BEFORE: {json.dumps(change["before"], indent=4)}')
        print(f'  AFTER : {json.dumps(change["after"],  indent=4)}')
    else:
        print(f'  value: {change}')
    print()

In [ ]:
# Also diff adjacent steps 0 → 1 for a tighter view
diff_01 = await debugger.diff('wf-001', step_a=0, step_b=1)

print('=== Diff: step 0 → step 1 (what data_clean added) ===')
print()

for field, change in diff_01.items():
    print(f'Field: {field!r}')
    if isinstance(change, dict) and 'before' in change:
        before_keys = list(change['before'].keys()) if isinstance(change['before'], dict) else change['before']
        after_keys  = list(change['after'].keys())  if isinstance(change['after'],  dict) else change['after']
        print(f'  BEFORE keys: {before_keys}')
        print(f'  AFTER  keys: {after_keys}')
        new_keys = [k for k in (after_keys if isinstance(after_keys, list) else [])
                    if k not in (before_keys if isinstance(before_keys, list) else [])]
        if new_keys:
            print(f'  NEW keys added by step 1: {new_keys}')
    else:
        print(f'  {change}')
    print()

## Section 6 — Replay from a Step with a Patched Agent

### The scenario

Imagine `data_clean` has a bug: it returns the clean rows under the wrong key
`cleaned` instead of `clean_rows`.  As a result, `data_analyze` sees no rows and
produces a mean of `0.0` over an empty list (or worse, a division error).

With time-travel debugging the fix workflow is:

1. Identify the bad step (`data_clean`, step 1).
2. Write a patched agent that returns the correct key.
3. Call `replay_from` starting at step 1 — the context from step 0 is automatically
   restored, so `data_fetch` is NOT re-run.
4. Confirm the downstream `data_analyze` output is now correct.

### How `replay_from` works

- Restores the `context` snapshot captured at `from_step - 1`.
- For each step from `from_step` onwards:
  - If the step's agent name appears in the `agents=` dict, runs the replacement.
  - Otherwise, **freezes** the original output (carries it forward unchanged).
- Saves new snapshots under a `replay` workflow ID so the original is untouched.
- Returns the list of outputs produced by the replayed steps.

The `agents` dict is keyed by the **Python class name** of the agent
(`"FunctionAgent"` in our case), not the agent's `name` attribute.
This lets you swap out all agents of a given type, or swap a specific named agent.

Wait — actually, let's verify the keying by looking at how `replay_from` resolves
replacements.  The debugger stores `agent=type(agent).__name__` in the snapshot,
and then matches the replacement dict against that.  Since all three steps use
`FunctionAgent`, we must target only step 1.  We do this by providing the replacement
only for the step we want to patch; the others will freeze automatically.

In [ ]:
# ── Step 1: introduce the bug ────────────────────────────────────────────────
# Run a second workflow with a buggy data_clean that uses the wrong output key

buggy_clean_agent = FunctionAgent(
    'data_clean',
    fn=lambda ctx: {
        # BUG: key is 'cleaned' instead of 'clean_rows'
        # data_analyze reads 'clean_rows', so it will see an empty list
        'cleaned': [
            r for r in ctx.get('raw_rows', [])
            if r.get('value') is not None and r.get('flag') == 'ok'
        ],
        'dropped_count': 0,
    }
)

buggy_steps = [
    (data_fetch_agent,   {'threshold': 5.0}),
    (buggy_clean_agent,  {}),
    (data_analyze_agent, {}),
]

buggy_results = await debugger.run_chain(buggy_steps, workflow_id='wf-buggy')

print('Buggy pipeline outputs:')
for i, out in enumerate(buggy_results):
    print(f'  step {i}: {out}')

print()
print('Notice: data_analyze sees count=0, mean=0 because clean_rows was empty.')
print('The bug is in step 1 — wrong key name.')

In [ ]:
# ── Step 2: write the patched agent ──────────────────────────────────────────
# The fix is trivial: return 'clean_rows' instead of 'cleaned'

patched_clean_agent = FunctionAgent(
    'data_clean',
    fn=lambda ctx: {
        'clean_rows': [
            r for r in ctx.get('raw_rows', [])
            if r.get('value') is not None and r.get('flag') == 'ok'
        ],
        'dropped_count': sum(
            1 for r in ctx.get('raw_rows', [])
            if r.get('value') is None or r.get('flag') != 'ok'
        ),
    }
)

# ── Step 3: replay from step 1 with the patched agent ────────────────────────
# The context at step 0 (data_fetch outputs) is restored automatically.
# We replace the 'FunctionAgent' at step 1 with patched_clean_agent.
# Step 2 (data_analyze) has no replacement, so it will re-run with frozen
# context UNLESS we also provide it.  Here we want it to re-run with the
# correct clean_rows, so we pass data_analyze_agent as well.

replay_results = await debugger.replay_from(
    'wf-buggy',
    from_step=1,
    agents={
        'FunctionAgent': patched_clean_agent,   # replaces ALL FunctionAgent steps at >= step 1
    },
)

print('Replay outputs (steps 1 and 2):')
for i, out in enumerate(replay_results, start=1):
    print(f'  step {i}: {out}')

print()
print('Now data_analyze correctly sees the clean rows and computes real statistics.')
print('data_fetch (step 0) was NOT re-run — its output was restored from the snapshot.')

In [ ]:
# Verify: compare the original (buggy) analysis vs the replayed (fixed) analysis

buggy_analyze_snap = await debugger.get_snapshot('wf-buggy', step=2)

# The replay is stored under a new workflow ID — find it
# replay_from saves under "<workflow_id>:replay:<from_step>:<hex>"
# We can find it by listing all workflows stored in the in-memory store
all_snaps_in_store = list(debugger.store._store.values())
replay_wf_ids = set(
    s.workflow_id for s in all_snaps_in_store
    if s.workflow_id.startswith('wf-buggy:replay')
)
print(f'Replay workflow IDs created: {replay_wf_ids}')

replay_wf_id = list(replay_wf_ids)[0]
replay_analyze_snap = await debugger.get_snapshot(replay_wf_id, step=2)

print()
print('Original buggy data_analyze output:')
print(f'  {buggy_analyze_snap.output}')
print()
print('Replayed (fixed) data_analyze output:')
print(f'  {replay_analyze_snap.output}')
print()
print('The fix works: mean is now computed over real rows, not an empty list.')

## Section 7 — Replay with Patched Input Parameters

Sometimes you don't want to change agent code — you want to change an **input
parameter**.  For example: what if we re-ran the analysis with a stricter threshold
of `0.9` instead of `5.0`?

The `patch=` argument to `replay_from` injects extra key/value pairs into the
restored context before every replayed step.  The patched values take precedence over
whatever was in the original snapshots.

Here we replay the good `wf-001` pipeline from step 2 (`data_analyze`) with
`threshold=0.9`.  Since the mean is `~5.15`, the threshold check should still pass
for `0.9` but the numerical result will differ.

We can also replay from step 2 alone (skipping data_fetch and data_clean entirely)
because their outputs are frozen from the original snapshots.

In [ ]:
# Replay the good workflow from step 2 with a different threshold
# No agent replacement — data_analyze runs unchanged but sees threshold=0.9

replay_patched = await debugger.replay_from(
    'wf-001',
    from_step=2,
    agents={},                    # no agent replacement
    patch={'threshold': 0.9},     # inject new threshold into context
)

print('Replay from step 2 with threshold=0.9')
print(f'  Output: {replay_patched[0]}')

# Compare with original step 2 output
original_snap2 = await debugger.get_snapshot('wf-001', step=2)
print()
print(f'Original step 2 output (threshold=5.0): {original_snap2.output}')
print()
print('Observation: threshold_pass is True in both cases because mean > 0.9.')
print('But try threshold=100.0 to see a failing threshold:')

replay_high_thresh = await debugger.replay_from(
    'wf-001',
    from_step=2,
    agents={},
    patch={'threshold': 100.0},
)
print(f'  threshold=100.0 replay output: {replay_high_thresh[0]}')

## Section 8 — SQLite Persistence

The `InMemorySnapshotStore` is convenient but ephemeral — all snapshots are lost
when the Python process ends.  For **cross-session debugging** (e.g., a production
pipeline ran overnight and you want to inspect it the next morning),
use `SQLiteSnapshotStore`.

`SQLiteSnapshotStore(path)` creates a SQLite database at the given path.  All
snapshot data is JSON-serialised and stored in a single `snapshots` table with an
index on `(workflow_id, step_index)`.

### Workflow

1. First session: run the pipeline with an `SQLiteSnapshotStore`; snapshots are
   written to `debug.db`.
2. Second session (new kernel): open the same `debug.db` and inspect snapshots
   without re-running anything.

In [ ]:
import os

DB_PATH = '/tmp/multigen_debug.db'

# Remove any pre-existing database so we start clean for this demo
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

# ── Session 1: run the pipeline and save to SQLite ───────────────────────────

sqlite_store = SQLiteSnapshotStore(DB_PATH)
debugger_sqlite = WorkflowDebugger(store=sqlite_store)

sqlite_steps = [
    (data_fetch_agent,   {'threshold': 5.0}),
    (data_clean_agent,   {}),
    (data_analyze_agent, {}),
]

await debugger_sqlite.run_chain(sqlite_steps, workflow_id='wf-sqlite-001')

db_size = os.path.getsize(DB_PATH)
print(f'Pipeline complete.  SQLite database written to: {DB_PATH}')
print(f'Database file size: {db_size} bytes')
print()

# ── Session 2: reload from disk in a new WorkflowDebugger ────────────────────
# (In a real scenario this would be a new Python process / Jupyter kernel)

reloaded_store    = SQLiteSnapshotStore(DB_PATH)           # re-opens same file
reloaded_debugger = WorkflowDebugger(store=reloaded_store) # no run_chain needed

reloaded_steps = await reloaded_debugger.list_steps('wf-sqlite-001')

print(f'Reloaded {len(reloaded_steps)} snapshots from disk:')
for snap in reloaded_steps:
    print(f'  step {snap.step_index}  node={snap.node_id!r:<14}  status={snap.status!r}')
    print(f'           output: {snap.output}')

print()
print('SQLite persistence confirmed: all snapshots survived a simulated kernel restart.')

In [ ]:
# You can perform all debugger operations on the reloaded store
# without having re-run the pipeline

reloaded_snap1 = await reloaded_debugger.get_snapshot('wf-sqlite-001', step=1)

print('Snapshot at step 1 loaded from SQLite:')
print(f'  node_id     : {reloaded_snap1.node_id}')
print(f'  status      : {reloaded_snap1.status}')
print(f'  duration_ms : {reloaded_snap1.duration_ms}')
print(f'  output      : {reloaded_snap1.output}')
print()

reloaded_diff = await reloaded_debugger.diff('wf-sqlite-001', step_a=0, step_b=2)
print(f'Diff step 0→2 from SQLite has {len(reloaded_diff)} changed fields.')
print('Fields that changed:', list(reloaded_diff.keys()))

## Section 9 — Exporting a Full Trace

`await debugger.export_trace(workflow_id)` returns a list of dicts — one per step —
that is fully JSON-serialisable.  This makes it easy to:

- Write the trace to a file for archiving.
- POST it to an observability backend (Jaeger, Zipkin, Honeycomb, Datadog).
- Feed it into a custom UI or notebook visualization.
- Compare runs using standard JSON diff tools.

Each dict corresponds to `Snapshot.to_dict()` and contains all fields including
`snapshot_id` (a UUID for deduplication) and `metadata` (extensible key-value pairs).

In [ ]:
trace = await debugger.export_trace('wf-001')

print(f'Trace contains {len(trace)} step records.')
print()

# Show each step's summary fields
for record in trace:
    print(f"Step {record['step_index']}  node={record['node_id']!r:<14}  "
          f"agent={record['agent']!r:<15}  status={record['status']!r}  "
          f"duration={record['duration_ms']:.2f}ms")

print()
print('Full first step record (JSON):')
print(json.dumps(trace[0], indent=2))

In [ ]:
# Save trace to a JSON file — ready for any observability tool

TRACE_PATH = '/tmp/wf_001_trace.json'

with open(TRACE_PATH, 'w') as f:
    json.dump(trace, f, indent=2)

print(f'Trace written to: {TRACE_PATH}')
print(f'File size: {os.path.getsize(TRACE_PATH)} bytes')
print()

# Demonstrate round-trip: reload and verify step count
with open(TRACE_PATH) as f:
    reloaded_trace = json.load(f)

print(f'Reloaded trace: {len(reloaded_trace)} steps')
print('Step node IDs:', [r['node_id'] for r in reloaded_trace])
print()

# Example: compute total pipeline latency from the trace
total_ms = sum(r['duration_ms'] for r in reloaded_trace)
print(f'Total pipeline latency (sum of step durations): {total_ms:.2f} ms')
print()
print('This trace JSON can be posted to any OpenTelemetry-compatible backend:')
print('  import requests')
print('  requests.post("http://jaeger:4317/traces", json=trace)')

## Section 10 — Injecting a Failure and Replaying with a Fix

This section demonstrates the full debugging cycle:

1. An agent raises an exception **conditionally** (only when a specific input value
   is present — mimicking a real data-dependent bug).
2. The `WorkflowDebugger` captures the failure as a snapshot with `status="failed"`
   and the exception message in the `error` field.
3. We inspect the failure snapshot to understand what input caused the crash.
4. We write a robust replacement agent that handles the edge case.
5. We replay from the failed step and confirm the fixed output.

### The scenario

The `data_analyze` agent crashes when `clean_rows` is empty — `sum([]) / 0` raises
`ZeroDivisionError`.  This happens when **all** rows are filtered out by `data_clean`.

In [ ]:
# ── Agents that trigger a conditional failure ─────────────────────────────────

# data_fetch returns only rows with missing values — all will be dropped
all_null_fetch_agent = FunctionAgent(
    'data_fetch',
    fn=lambda ctx: {
        'raw_rows': [
            {'id': 1, 'value': None, 'flag': 'ok'},
            {'id': 2, 'value': None, 'flag': 'ok'},
        ],
        'source': 'mock_db_empty',
        'record_count': 2,
    }
)

# data_clean will produce an empty clean_rows list
# data_analyze will crash with ZeroDivisionError

crashing_analyze_agent = FunctionAgent(
    'data_analyze',
    fn=lambda ctx: {
        # BUG: no guard against empty list — crashes with ZeroDivisionError
        'mean':  sum(r['value'] for r in ctx['clean_rows']) / len(ctx['clean_rows']),
        'count': len(ctx['clean_rows']),
    }
)

failing_steps = [
    (all_null_fetch_agent,   {}),
    (data_clean_agent,       {}),
    (crashing_analyze_agent, {}),
]

failing_results = await debugger.run_chain(failing_steps, workflow_id='wf-failing')

print('Pipeline outputs (chain stops at first failure):')
for i, out in enumerate(failing_results):
    print(f'  step {i}: {out}')

In [ ]:
# ── Inspect the failure snapshot ──────────────────────────────────────────────

failing_steps_list = await debugger.list_steps('wf-failing')

print('All captured snapshots for wf-failing:')
for snap in failing_steps_list:
    marker = '💥' if snap.status == 'failed' else '✓'
    print(f'  {marker} step {snap.step_index}  node={snap.node_id!r:<15}  status={snap.status!r}')
    if snap.error:
        print(f'       error: {snap.error}')

print()

# Inspect the failing step in detail
failed_snap = await debugger.get_snapshot('wf-failing', step=2)

print('=== Failure snapshot at step 2 ===')
print(f'  node_id     : {failed_snap.node_id}')
print(f'  status      : {failed_snap.status}')
print(f'  error       : {failed_snap.error}')
print(f'  input_params: {json.dumps(failed_snap.input_params, indent=4)}')
print()
print('Root cause: clean_rows is empty — division by zero in mean calculation.')

In [ ]:
# ── Fix: a robust analyze agent that handles empty input ─────────────────────

robust_analyze_agent = FunctionAgent(
    'data_analyze',
    fn=lambda ctx: (
        {'mean': None, 'count': 0, 'max': None, 'min': None, 'threshold_pass': False,
         'warning': 'No rows to analyze after cleaning'}
        if not ctx.get('clean_rows')
        else {
            'mean':  round(sum(r['value'] for r in ctx['clean_rows']) / len(ctx['clean_rows']), 4),
            'count': len(ctx['clean_rows']),
            'max':   max(r['value'] for r in ctx['clean_rows']),
            'min':   min(r['value'] for r in ctx['clean_rows']),
            'threshold_pass': (
                sum(r['value'] for r in ctx['clean_rows']) / len(ctx['clean_rows'])
            ) >= ctx.get('threshold', 5.0),
        }
    )
)

# ── Replay from step 2 with the fixed agent ───────────────────────────────────
# Steps 0 and 1 are not re-run — their outputs are frozen from wf-failing.
# Step 2 is replaced with robust_analyze_agent.

fixed_replay = await debugger.replay_from(
    'wf-failing',
    from_step=2,
    agents={'FunctionAgent': robust_analyze_agent},
)

print('Fixed replay output at step 2:')
print(json.dumps(fixed_replay[0], indent=2))
print()
print('The fixed agent gracefully handles empty input.')
print('data_fetch and data_clean were NOT re-run — only step 2 was replayed.')

## Section 11 — Multi-Workflow Debugging

A single `WorkflowDebugger` (and its underlying store) can hold snapshots for many
independent workflows simultaneously.  Snapshots are always namespaced by
`workflow_id` — `list_steps('wf-A')` will never return steps from `wf-B`.

This is important in production environments where the same debugger instance
may be tracking dozens of concurrent workflow runs.

This section:
1. Runs two distinct workflows (`wf-alpha` and `wf-beta`) through the same debugger.
2. Verifies that `list_steps` is scoped correctly.
3. Shows that diffs and snapshots are also scoped.

In [ ]:
# Create a fresh debugger with a clean store so we don't mix in earlier workflows
multi_debugger = WorkflowDebugger()

# ── Workflow wf-alpha: processes text data ────────────────────────────────────

alpha_tokenize = FunctionAgent(
    'tokenize',
    fn=lambda ctx: {'tokens': ctx.get('text', '').split(), 'token_count': len(ctx.get('text', '').split())}
)
alpha_score = FunctionAgent(
    'score',
    fn=lambda ctx: {'score': len(ctx.get('tokens', [])) * 1.5, 'model': 'tf-idf-mock'}
)

await multi_debugger.run_chain(
    [
        (alpha_tokenize, {'text': 'hello world this is workflow alpha'}),
        (alpha_score,    {}),
    ],
    workflow_id='wf-alpha',
)

# ── Workflow wf-beta: processes numeric data ──────────────────────────────────

beta_generate = FunctionAgent(
    'generate',
    fn=lambda ctx: {'numbers': list(range(ctx.get('start', 1), ctx.get('end', 6))), 'count': ctx.get('end', 6) - ctx.get('start', 1)}
)
beta_square = FunctionAgent(
    'square',
    fn=lambda ctx: {'squared': [x**2 for x in ctx.get('numbers', [])], 'total': sum(x**2 for x in ctx.get('numbers', []))}
)
beta_report = FunctionAgent(
    'report',
    fn=lambda ctx: {'summary': f"Squared {ctx.get('count', 0)} numbers, total={ctx.get('total', 0)}", 'workflow': 'beta'}
)

await multi_debugger.run_chain(
    [
        (beta_generate, {'start': 1, 'end': 6}),
        (beta_square,   {}),
        (beta_report,   {}),
    ],
    workflow_id='wf-beta',
)

print('Both workflows completed.  Total snapshots in store:', len(multi_debugger.store))

In [ ]:
# ── Verify isolation: list_steps only returns the right workflow's steps ───────

alpha_steps = await multi_debugger.list_steps('wf-alpha')
beta_steps  = await multi_debugger.list_steps('wf-beta')

print(f'wf-alpha steps: {len(alpha_steps)}')
for s in alpha_steps:
    print(f'  step {s.step_index}  node={s.node_id!r}  workflow_id={s.workflow_id!r}')

print()
print(f'wf-beta steps: {len(beta_steps)}')
for s in beta_steps:
    print(f'  step {s.step_index}  node={s.node_id!r}  workflow_id={s.workflow_id!r}')

print()
assert all(s.workflow_id == 'wf-alpha' for s in alpha_steps), 'wf-alpha isolation violated!'
assert all(s.workflow_id == 'wf-beta'  for s in beta_steps),  'wf-beta isolation violated!'
print('Isolation verified: list_steps returns only the requested workflow\'s steps.')

In [ ]:
# ── Each workflow can be diffed independently ─────────────────────────────────

alpha_diff = await multi_debugger.diff('wf-alpha', step_a=0, step_b=1)
beta_diff  = await multi_debugger.diff('wf-beta',  step_a=0, step_b=2)

print('wf-alpha diff (step 0 → 1), changed fields:')
for field in alpha_diff:
    print(f'  {field}')

print()
print('wf-beta diff (step 0 → 2), changed fields:')
for field in beta_diff:
    print(f'  {field}')

print()
print('Each workflow maintains a completely independent snapshot timeline.')

# Show wf-beta's final context to confirm end-to-end state
beta_final = await multi_debugger.get_snapshot('wf-beta', step=2)
print()
print('wf-beta final context:')
print(json.dumps(beta_final.context, indent=2))

## Summary

### The WorkflowDebugger API at a glance

| Method | Signature | What it does |
|--------|-----------|-------------|
| `run_chain` | `(steps, workflow_id=)` | Run `(agent, params)` pairs sequentially, save a snapshot per step |
| `list_steps` | `(workflow_id)` | Return all snapshots sorted by step index |
| `get_snapshot` | `(workflow_id, step=N)` | Return the single snapshot at step N |
| `diff` | `(workflow_id, step_a=A, step_b=B)` | Return dict of changed fields between two steps |
| `replay_from` | `(workflow_id, from_step=N, agents={...}, patch={...})` | Replay from step N with optional agent replacements and input patches |
| `export_trace` | `(workflow_id)` | Return all snapshots as JSON-serialisable dicts |
| `reset` | `(workflow_id=None)` | Clear snapshots and counters |

### Storage backends

| Backend | Constructor | Persistence | Use case |
|---------|-------------|-------------|----------|
| `InMemorySnapshotStore` | `InMemorySnapshotStore()` | In-process only | Tests, notebooks, development |
| `SQLiteSnapshotStore` | `SQLiteSnapshotStore(path)` | Cross-session, single file | Local debugging, CI pipelines |

### Time-travel debugging workflow

```
1. Run pipeline with WorkflowDebugger → snapshots captured automatically
2. list_steps(wf_id) → find which step has wrong output
3. get_snapshot(wf_id, step=N) → inspect exact input/output/context
4. diff(wf_id, step_a=good, step_b=bad) → see exactly what changed
5. Write patched agent OR prepare patch={} dict
6. replay_from(wf_id, from_step=N, agents={...}, patch={...})
   → steps 0..N-1 are FROZEN (not re-run)
   → step N onwards re-runs with your fix
7. export_trace(wf_id) → ship to observability backend
```

### Next steps

- **Notebook 19**: Resilience patterns (Retry, CircuitBreaker, MemoryAgent)
- **Notebook 18**: Runtime — unified execution with simulator integration
- Combine `WorkflowDebugger` with `Runtime` to capture snapshots from real
  Temporal or Kafka-backed workflows.
- Build a custom UI on top of `export_trace` for visual step inspection.